# Object Detection Workflow via API

**How to evaluate common workflows via the RI API**

This notebook demonstrates a full end to end workflow of the RI backend. The general steps are:

* Create wrapper objects as necessary (not all wrappers are needed for every tool)
  * Create a model wrapper
  * Create a dataset wrapper
  * Create a metric wrapper
* Create configuration to define analyses to run
* Generate "Test Stage" object(s)
* Run analyses

All RI wrappers are maite-compliant (https://jatic.pages.jatic.net/cdao/maite/). 

**IMPORTANT**:   
* **Since we are using mini test datasets, the values presented in the results may not have significant meaning.** Use full datasets with models trained on similar data to view accurate results. 
* This notebook requires a dev install since it utilizes data from the test suite.

In [ ]:
from jatic_ri import PACKAGE_DIR, cache_path
from jatic_ri.util.utils import create_expandable_output

import json
import logging
import os 
from pathlib import Path
import torch
import warnings

logger = logging.getLogger(__name__)

DEV_INSTALL_ROOT = PACKAGE_DIR.parent.parent
EXAMPLE_DATA_DIR = PACKAGE_DIR.parents[1].joinpath('tests/testing_utilities/example_data')
EXAMPLE_COCO_DATASET_DIR = EXAMPLE_DATA_DIR.joinpath('coco_resized_val2017').resolve()
EXAMPLE_YOLO_DATASET_DIR = DEV_INSTALL_ROOT.joinpath('tests/test_object_detection/data/yolo_dataset').resolve()
EXAMPLE_VISDRONE_DATASET_DIR = EXAMPLE_DATA_DIR.joinpath('visdrone_dataset')

warnings.filterwarnings("ignore", category=UserWarning)

### Set cache path 

In [ ]:
# the RI has a default output cache path, but it can be overriden as needed. The cache will only be written if `use_stage_cache=True` on the test stage.
print(f'The default cache path is {cache_path()}')

# set the cache path
cache_path('tscache')

print(f'The current cache path is {cache_path()}')

### Apply RealLabel patch for malformed bounding boxes

In [ ]:
from reallabel.maite_reallabel import MAITERealLabel
import numpy as np 
from unittest.mock import patch

def patched_convert_bbox_format(xyxy_bbox):
    x1 = xyxy_bbox[0]
    y1 = xyxy_bbox[1]
    x2 = xyxy_bbox[2]
    y2 = xyxy_bbox[3]

    if x1 >= x2:
        x2 = x1 + 1
        logger.warning("Bounding box misformed, x1 >= x2, resetting as x2 = x1 + 1 to avoid errors in analysis computations.")
    if y1 >= y2:
        y2 = y1 + 1
        logger.warning("Bounding box misformed, y1 >= y2, resetting as y2 = y1 + 1 to avoid errors in analysis computations.")

    return [
        x1,
        y1,
        x2 - x1,
        y2 - y1,
    ]
    
patch.object(MAITERealLabel, '_convert_bbox_format', patched_convert_bbox_format).start();

### Define task 

In [ ]:
task = 'object_detection' 

### Create a model wrapper

Model wrappers hold the model object, weights and metadata.

The RI has two object detection model wrappers - `torchvision` and `visdrone`. Below are examples of each of the available configuration options. 

#### Torchvision OD Model using default weights from Torchvision

In [ ]:
from jatic_ri.object_detection.models import TorchvisionODModel
model_name = "ssdlite320_mobilenet_v3_large"
model_wrapper = TorchvisionODModel(model_name=model_name)

#### Torchvision OD Model using custom weights and config

Note: The configuration file is a JSON formatted text file that contains `index2label` as a key with its value being a dictionary of {index: category label} for the model e.g. `{0: '__background__', 1: 'person', 2: 'bicycle' ...}`.

In [ ]:
# save metadata and state_dict from previous cell to disk
config_path = cache_path() / "my_model.json"
pickle_path = cache_path() / "my_model.pt"

os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    json.dump({"index2label": model_wrapper.index2label}, f)
_ = torch.save(model_wrapper.model.state_dict(), pickle_path)

# extra kwargs to pass through to torchvision model object
kwargs = {}
torch_model_wrapper = TorchvisionODModel(
    model_name=model_name,
    weights_path=pickle_path,
    config_path=config_path,
    model_id="arbitraryidnumber",
    **kwargs,
)

#### VisDrone OD Model using weights from Kitware's data server

Visdrone architecture options are `resnet18`, `resnet50`, and `res2net50`. Custom weights on these models are not currently supported. 

In [ ]:
from jatic_ri.object_detection.models import VisdroneODModel

# specify directory for the model weights to be downloaded
download_weights_dir = 'visdrone_model_weights'
visdrone_model_wrapper = VisdroneODModel(
    arch='resnet18',
    model_pickle_dir=download_weights_dir,
    model_id='1234',
)

### Create dataset wrapper

Dataset wrappers control the access to the dataset and contain metadata about the dataset and about individual images.

The RI has three dataset wrappers for object detection - `coco`, `yolo` and `visdrone`. Below are examples of loading each type. 

#### COCO dataset

In [ ]:
from jatic_ri.object_detection.datasets import CocoDetectionDataset

coco_dataset_wrapper = CocoDetectionDataset(
    root=str(EXAMPLE_COCO_DATASET_DIR),
    ann_file=str(EXAMPLE_COCO_DATASET_DIR.joinpath('instances_val2017_resized_6.json')),
)

coco_dataset_wrapper_single = CocoDetectionDataset(
    root=str(EXAMPLE_COCO_DATASET_DIR),
    ann_file=str(EXAMPLE_COCO_DATASET_DIR.joinpath('single_image.json')),
)

#### YOLO dataset

In [ ]:
from jatic_ri.object_detection.datasets import YoloDetectionDataset

CURDIR = os.getcwd()
os.chdir(DEV_INSTALL_ROOT) # the yolo test metadata file is written relative to the root pkg dir

yolo_dataset_wrapper = YoloDetectionDataset(
    yaml_dataset=str(EXAMPLE_YOLO_DATASET_DIR.joinpath('dataset.yaml')),
    ann_dir=str(EXAMPLE_YOLO_DATASET_DIR.joinpath('ann_dir')),
)
os.chdir(CURDIR)

#### Visdrone dataset 

In [ ]:
from jatic_ri.object_detection.datasets import VisdroneDetectionDataset

visdrone_dataset_wrapper = VisdroneDetectionDataset(
    root=str(EXAMPLE_VISDRONE_DATASET_DIR),
)

### Create metric wrapper 

Metric wrappers provide standardized access to metric algorithms across the pydata ecosystem. 

The RI has one object detection metric wrapper - `mean average precision (mAP)`. 

In [ ]:
from jatic_ri.object_detection.metrics import map50_torch_metric_factory
metric_wrapper = map50_torch_metric_factory()

### Instantiate and run each Test Stage

Each stage will be run and a list of slide data will be built

In [ ]:
# collect all the report slides in a list (to be convert to ppt after all stages are run)
slides = []

#### MAITE - Baseline evaluation 

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import BaselineEvaluation

# instantiate test stage object from config values
test_stage = BaselineEvaluation()
# populate test_stage with data
test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_threshold(0.5)
test_stage.load_model(visdrone_model_wrapper, model_id='model1')

# run analysis for this test stage
base_eval_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
base_eval_run.outputs

#### NRTK 

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import NRTKTestStage, NRTKTestStageConfig

nrtk_config_dict = {
    'name': 'natural_robustness_TestFactory',
    'perturber_factory': {
        'type': 'nrtk.impls.perturb_image_factory.generic.one_step.OneStepPerturbImageFactory',
        'nrtk.impls.perturb_image_factory.generic.one_step.OneStepPerturbImageFactory': {
            'perturber': 'nrtk.impls.perturb_image.generic.PIL.enhance.BrightnessPerturber',
            'theta_key': 'factor',
            'theta_value': 10.0,
        },
    },
}
nrtk_config = NRTKTestStageConfig(**nrtk_config_dict)
# instantiate test stage object from config values
test_stage = NRTKTestStage(nrtk_config)
# populate test_stage with data
test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_threshold(0.5)
test_stage.load_model(visdrone_model_wrapper, model_id='model1')

# run analysis for this test stage
nrtk_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
nrtk_run.outputs

#### XAITK

XAITK should only be run on single images or a datasets with only a few images due to memory constraints

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import XAITKTestStage, XAITKConfigOD

xaitk_config_dict = {
    'name': 'saliency_XAITKApp_0',
    'saliency_generator': {
        'type': 'xaitk_saliency.impls.gen_object_detector_blackbox_sal.drise.DRISEStack',
        'xaitk_saliency.impls.gen_object_detector_blackbox_sal.drise.DRISEStack': {
            'n': 20,
            's': 7,
            'p1': 0.7,
            'seed': 42,
            'threads': 8,
            'fill': [95, 96, 93],
        },
    },
    'img_batch_size': 1,
}
xaitk_config = XAITKConfigOD(**xaitk_config_dict)
# instantiate test stage object from config values
test_stage = XAITKTestStage(xaitk_config)
# populate test_stage with data
# use single image dataset since XAITK is computationally intense
test_stage.load_dataset(coco_dataset_wrapper_single, dataset_id='dataset1')
test_stage.load_model(visdrone_model_wrapper, model_id='model1')

# run analysis for this test stage
xaitk_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# XAITK produces one slide per object per image. 
# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(xaitk_run.outputs.results)

#### Survivor 

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import SurvivorTestStage, SurvivorConfig

survivor_config_dict = {
    'metric_column': 'metric',
    'conversion_type': 'original',
    'otb_threshold': 0.5,
    'easy_hard_threshold': 0.5,
}
survivor_config = SurvivorConfig(**survivor_config_dict)
# instantiate test stage object from config values
test_stage = SurvivorTestStage(survivor_config)
# populate test_stage with data
test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_models({'model1': visdrone_model_wrapper, 'model2': model_wrapper})

# run analysis for this test stage
survivor_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
create_expandable_output(survivor_run.outputs)

#### Dataeval - Bias

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import DatasetBiasTestStage

# instantiate test stage object
test_stage = DatasetBiasTestStage()
# the device type can be changed directly on the stage
test_stage.device = "cpu"
# populate test_stage with data
test_stage.load_dataset(coco_dataset_wrapper, dataset_id='dataset1')

# run analysis for this test stage
bias_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

In [ ]:
# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(bias_run.outputs.balance)

In [ ]:
create_expandable_output(bias_run.outputs.coverage)

In [ ]:
create_expandable_output(bias_run.outputs.diversity)

#### Dataeval - Feasibility

NOTE: Currently non-functional ([RI Issue #181](https://gitlab.jatic.net/jatic/reference-implementation/reference-implementation/-/issues/181))

In [ ]:
# %%time
# from jatic_ri.object_detection.test_stages import DatasetFeasibilityTestStage

# # instantiate test stage object 
# test_stage = DatasetFeasibilityTestStage()
# # populate test_stage with data
# test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')
# test_stage.load_threshold(0.5)

# # run analysis for this test stage
# feasibility_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# # collect the slides for the final report
# stage_slides = test_stage.collect_report_consumables()
# # add to overall slide list
# slides += stage_slides

# # view results
# feasibility_run.outputs

#### Dataeval - Cleaning

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import DatasetCleaningTestStage

# instantiate test stage object
test_stage = DatasetCleaningTestStage()
# populate test_stage with data
test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')

# run analysis for this test stage
cleaning_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
create_expandable_output(cleaning_run.outputs)

#### Dataeval - Shift

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import DatasetShiftTestStage

# instantiate test stage object
test_stage = DatasetShiftTestStage()
# populate test_stage with data
test_stage.load_datasets(dataset_1=coco_dataset_wrapper, dataset_1_id='dataset1', dataset_2=coco_dataset_wrapper, dataset_2_id='dataset2')

# run analysis for this test stage
shift_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(shift_run.outputs)

#### Reallabel

In [ ]:
%%time
from jatic_ri.object_detection.test_stages import RealLabelTestStage, RealLabelConfig

# define reallabel config
reallabel_config_dict = {
    'additional_outputs': [
        'sequence_priority_score',
        'sequence_priority_score_balanced',
        'classification_disagreements',
        'wanrs',
        'aggregated_confidence_df',
    ],
    'run_with_ground_truth': True,
    'deduplication_iou_threshold': 0.5,
    'threshold_max_aggregated_confidence_fp': 0.5,
    'threshold_min_aggregated_confidence_fn': 0.5,
    'use_thresholds': True,
    'class_agnostic': True,
    'run_confidence_calibration': False,
    'column_names': {
        'unique_identifier_columns': ['id'],
        'calibrated_confidence_column': 'score',
    },
}
reallabel_config = RealLabelConfig(**reallabel_config_dict)
# instantiate test stage object from config values
test_stage = RealLabelTestStage(reallabel_config)
# populate test_stage with data
test_stage.load_dataset(visdrone_dataset_wrapper, dataset_id='dataset1')
test_stage.load_models({'model1': visdrone_model_wrapper, 'model2': visdrone_model_wrapper})

# run analysis for this test stage
reallabel_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(reallabel_run.outputs.results)

## Construct final report

In [ ]:
from gradient.templates_and_layouts.create_deck import create_deck

# construct ppt report with summarized results
report_path = cache_path() / "report"
report_path.mkdir(parents=True, exist_ok=True)
report_filename = 'sample_report'
report = create_deck(slides, report_path, deck_name=report_filename)

print(f'Report saved to {report}')

<hr>